# Transformación de Bronze a Silver

Aplica las reglas aprobadas de `config/silver.json` a las tablas Bronze. Ejecutar después de sincronizar los lotes. Reconstruye únicamente las tablas Silver derivadas; no modifica Bronze ni sus checkpoints. No resuelve versiones ni ejecuta `MERGE`.


In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(project_root / 'scripts'))
from silver_quality import transform_publicaciones, transform_mediciones
silver = json.loads((project_root / 'config' / 'silver.json').read_text(encoding='utf-8'))
bronze = json.loads((project_root / silver['bronze_config']).read_text(encoding='utf-8'))
print('Origen:', bronze['tables'])
print('Destino:', silver['tables'])


Origen: {'publicaciones': 'workspace.lensdata_bronze.publicaciones', 'mediciones': 'workspace.lensdata_bronze.mediciones'}
Destino: {'publicaciones': 'workspace.lensdata_silver.publicaciones', 'mediciones': 'workspace.lensdata_silver.mediciones', 'rechazados': 'workspace.lensdata_silver.rechazados'}


## Validación y separación de registros

Las conversiones seguras convierten entradas no válidas en `NULL` y registran el motivo. Bronze debe permanecer como evidencia original para futuras comparaciones o cambios en la estructura de las capas posteriores.


In [2]:
raw_publicaciones = spark.table(bronze['tables']['publicaciones'])
raw_mediciones = spark.table(bronze['tables']['mediciones'])
publicaciones_ok, publicaciones_rechazadas = transform_publicaciones(raw_publicaciones, silver)
mediciones_ok, mediciones_rechazadas = transform_mediciones(raw_mediciones, publicaciones_ok, silver)
rechazados = publicaciones_rechazadas.unionByName(mediciones_rechazadas)

count_publicaciones = raw_publicaciones.count()
count_mediciones = raw_mediciones.count()
assert publicaciones_ok.count() + publicaciones_rechazadas.count() == count_publicaciones
assert mediciones_ok.count() + mediciones_rechazadas.count() == count_mediciones
print('Publicaciones Bronze / válidas / rechazadas:', count_publicaciones, publicaciones_ok.count(), publicaciones_rechazadas.count())
print('Mediciones Bronze / válidas / rechazadas:', count_mediciones, mediciones_ok.count(), mediciones_rechazadas.count())
display(rechazados.withColumn('reason_code', F.explode('reason_codes')).groupBy('entity', 'reason_code').count())


Publicaciones Bronze / válidas / rechazadas: 40 40 0
Mediciones Bronze / válidas / rechazadas: 473 469 4


,entity,reason_code,count
0,mediciones,UNKNOWN_VIDEO,1
1,mediciones,NEGATIVE_METRIC,1
2,mediciones,REQUIRED_FIELD,1
3,mediciones,INVALID_NUMBER,1


## Publicación de tablas Silver

La escritura se realiza solo tras validar que ninguna fila Bronze desaparece entre el conjunto válido y los rechazos. Se sobrescriben las tablas derivadas de Silver en cada ejecución; Bronze queda intacto.


In [3]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze['catalog']}.{silver['schema']}")
for dataframe, table in ((publicaciones_ok, silver['tables']['publicaciones']),
                         (mediciones_ok, silver['tables']['mediciones']),
                         (rechazados, silver['tables']['rechazados'])):
    (dataframe.write.format('delta').mode('overwrite')
     .option('overwriteSchema', 'true').saveAsTable(table))
print('Tablas Silver reconstruidas:', silver['tables'])


Tablas Silver reconstruidas: {'publicaciones': 'workspace.lensdata_silver.publicaciones', 'mediciones': 'workspace.lensdata_silver.mediciones', 'rechazados': 'workspace.lensdata_silver.rechazados'}
